# Suite CNF — Conflicting rules

A grid's rules are its children's adjacency specs, in the order they reached it. After each mutation it settles: the rules the mutation touched, a new one and the ignored, each in its turn, kept where it holds, met where one change of the sequence breaks no earlier rule that holds, and ignored, the later of the incompatible rules, where none can, until a mutation lets it be met. Each step is one mutation, a transaction expression the bench journals, and the container layout's adjustments to it, in order, which the step asserts; the suite's last case replays the journal in a fresh store, and finds the same adjustments. mbse-basis [CAP-240: conflicting layout rules ignored and requeued](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-240.md), for container layout, by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
from support import Bench, text
from mbse.Diagrams.Framework import Container, Logical

b = Bench()

<a id="CNF-01"></a>
## CNF-01 · A spec that would break an earlier rule that holds is ignored, and the earlier one kept

In [ ]:
assert b.child('', 'cpu') == '[{"kind":"grid","container":"overview","span":3,"grows":"rows","cell_rotation":0.0},{"kind":"place","shape":"cpu","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.layout('cpu', 3) == '[{"kind":"grid","container":"cpu","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('cpu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('cpu', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('cpu', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('cpu', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.child('cpu', 'e') == '[{"kind":"place","shape":"e","order":4,"to":[1,1]}]'
assert b.child('cpu', 'f') == '[{"kind":"place","shape":"f","order":5,"to":[1,2]}]'
assert b.spec('cpu', 'a', 'd', 180.0) == '[{"kind":"rule","from_shape":"a","to_shape":"d","degrees":180.0,"status":"held"}]'
assert b.spec('cpu', 'a', 'b', 270.0) == '[{"kind":"rule","from_shape":"a","to_shape":"b","degrees":270.0,"status":"ignored"}]'
assert b.spec('cpu', 'a', 'c', 270.0) == '[{"kind":"rule","from_shape":"a","to_shape":"c","degrees":270.0,"status":"ignored"}]'
assert b.cells("cpu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1],[[1,2],[["f",5]],null,1,1]]'
assert b.queue("cpu") == '[["a","b",270.0],["a","c",270.0]]'

<a id="CNF-02"></a>
## CNF-02 · A grid's rules are data: each spec, in the order it reached the grid, and whether it is ignored

In [ ]:
assert b.rules("cpu") == '[["a","d",180.0,false],["a","b",270.0,true],["a","c",270.0,true]]'

<a id="CNF-03"></a>
## CNF-03 · Removing the earlier spec lifts the conflict: the ignored rules are met, in the order they reached the grid

In [ ]:
assert b.unspec('cpu', 'a', 'd') == '[{"kind":"rule","from_shape":"a","to_shape":"d","degrees":180.0,"status":"dropped"},{"kind":"move","shape":"b","from":[0,1],"to":[0,0]},{"kind":"move","shape":"a","from":[0,0],"to":[0,1]},{"kind":"rule","from_shape":"a","to_shape":"b","degrees":270.0,"status":"met"},{"kind":"move","shape":"c","from":[0,2],"to":[0,0]},{"kind":"move","shape":"d","from":[1,0],"to":[0,2]},{"kind":"move","shape":"e","from":[1,1],"to":[1,0]},{"kind":"move","shape":"f","from":[1,2],"to":[1,1]},{"kind":"rule","from_shape":"a","to_shape":"c","degrees":270.0,"status":"met"}]'
assert b.cells("cpu") == '[[[0,0],[["b",1],["c",2]],"rows",2,1],[[0,1],[["a",0]],null,1,1],[[0,2],[["d",3]],null,1,1],[[1,0],[["e",4]],null,1,1],[[1,1],[["f",5]],null,1,1]]'
assert b.queue("cpu") == '[]'

<a id="CNF-04"></a>
## CNF-04 · Removing a child lifts a conflict its rules made, or one that closing its gap meets

In [ ]:
assert b.child('', 'gpu') == '[{"kind":"place","shape":"gpu","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.layout('gpu', 3) == '[{"kind":"grid","container":"gpu","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('gpu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('gpu', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('gpu', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('gpu', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.spec('gpu', 'a', 'd', 180.0) == '[{"kind":"rule","from_shape":"a","to_shape":"d","degrees":180.0,"status":"held"}]'
assert b.spec('gpu', 'a', 'b', 270.0) == '[{"kind":"rule","from_shape":"a","to_shape":"b","degrees":270.0,"status":"ignored"}]'
assert b.unchild('gpu', 'd') == '[{"kind":"leave","shape":"d","from":[1,0]},{"kind":"rule","from_shape":"a","to_shape":"d","degrees":180.0,"status":"dropped"},{"kind":"move","shape":"b","from":[0,1],"to":[0,0]},{"kind":"move","shape":"a","from":[0,0],"to":[0,1]},{"kind":"rule","from_shape":"a","to_shape":"b","degrees":270.0,"status":"met"},{"kind":"extents","rows":1,"columns":3}]'
assert b.cells("gpu") == '[[[0,0],[["b",1]],null,1,1],[[0,1],[["a",0]],null,1,1],[[0,2],[["c",2]],null,1,1]]'
assert b.queue("gpu") == '[]'
assert b.child('', 'lsu') == '[{"kind":"place","shape":"lsu","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.layout('lsu', 2) == '[{"kind":"grid","container":"lsu","span":2,"grows":"rows","cell_rotation":0.0}]'
assert b.child('lsu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('lsu', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('lsu', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[1,0]},{"kind":"extents","rows":2,"columns":2}]'
assert b.spec('lsu', 'b', 'c', 0.0) == '[{"kind":"move","shape":"c","from":[1,0],"to":[0,0]},{"kind":"move","shape":"a","from":[0,0],"to":[0,1]},{"kind":"move","shape":"b","from":[0,1],"to":[1,0]},{"kind":"rule","from_shape":"b","to_shape":"c","degrees":0.0,"status":"met"}]'
assert b.spec('lsu', 'b', 'a', 270.0) == '[{"kind":"rule","from_shape":"b","to_shape":"a","degrees":270.0,"status":"ignored"}]'
assert b.unchild('lsu', 'c') == '[{"kind":"leave","shape":"c","from":[0,0]},{"kind":"move","shape":"a","from":[0,1],"to":[0,0]},{"kind":"move","shape":"b","from":[1,0],"to":[0,1]},{"kind":"rule","from_shape":"b","to_shape":"c","degrees":0.0,"status":"dropped"},{"kind":"rule","from_shape":"b","to_shape":"a","degrees":270.0,"status":"met"},{"kind":"extents","rows":1,"columns":2}]'
assert b.cells("lsu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1]]'
assert b.queue("lsu") == '[]'

<a id="CNF-05"></a>
## CNF-05 · Adding children lifts a conflict with the grid's extents

In [ ]:
assert b.child('', 'mmu') == '[{"kind":"place","shape":"mmu","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.layout('mmu', 3) == '[{"kind":"grid","container":"mmu","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('mmu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('mmu', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.spec('mmu', 'a', 'b', 180.0) == '[{"kind":"rule","from_shape":"a","to_shape":"b","degrees":180.0,"status":"ignored"}]'
assert b.child('mmu', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('mmu', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,0]},{"kind":"move","shape":"c","from":[0,2],"to":[0,1]},{"kind":"move","shape":"d","from":[1,0],"to":[0,2]},{"kind":"move","shape":"b","from":[0,1],"to":[1,0]},{"kind":"rule","from_shape":"a","to_shape":"b","degrees":180.0,"status":"met"},{"kind":"extents","rows":2,"columns":3}]'
assert b.cells("mmu") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[0,2],[["d",3]],null,1,1],[[1,0],[["b",1]],null,1,1]]'

<a id="CNF-06"></a>
## CNF-06 · A child reverting out of a shared cell leaves the rules that hold in place

In [ ]:
assert b.child('', 'io') == '[{"kind":"place","shape":"io","order":4,"to":[1,1]}]'
assert b.layout('io', 3) == '[{"kind":"grid","container":"io","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('io', 'p') == '[{"kind":"place","shape":"p","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('io', 'q') == '[{"kind":"place","shape":"q","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('io', 'r') == '[{"kind":"place","shape":"r","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('io', 's') == '[{"kind":"place","shape":"s","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.child('io', 't') == '[{"kind":"place","shape":"t","order":4,"to":[1,1]}]'
assert b.spec('io', 'p', 'r', 180.0) == '[{"kind":"move","shape":"s","from":[1,0],"to":[0,2]},{"kind":"move","shape":"r","from":[0,2],"to":[1,0]},{"kind":"rule","from_shape":"p","to_shape":"r","degrees":180.0,"status":"met"}]'
assert b.spec('io', 'p', 't', 180.0) == '[{"kind":"move","shape":"t","from":[1,1],"to":[1,0]},{"kind":"rule","from_shape":"p","to_shape":"t","degrees":180.0,"status":"met"}]'
assert b.unspec('io', 'p', 'r') == '[{"kind":"rule","from_shape":"p","to_shape":"r","degrees":180.0,"status":"dropped"},{"kind":"move","shape":"r","from":[1,0],"to":[0,2]},{"kind":"move","shape":"s","from":[0,2],"to":[1,0]},{"kind":"move","shape":"t","from":[1,0],"to":[1,1]},{"kind":"move","shape":"t","from":[1,1],"to":[1,0]},{"kind":"move","shape":"s","from":[1,0],"to":[1,1]},{"kind":"rule","from_shape":"p","to_shape":"t","degrees":180.0,"status":"met"}]'
assert b.cells("io") == '[[[0,0],[["p",0]],null,1,1],[[0,1],[["q",1]],null,1,1],[[0,2],[["r",2]],null,1,1],[[1,0],[["t",4]],null,1,1],[[1,1],[["s",3]],null,1,1]]'

<a id="CNF-07"></a>
## CNF-07 · Removing an ignored spec moves nothing: it only drops the rule

In [ ]:
assert b.child('', 'dsp') == '[{"kind":"place","shape":"dsp","order":5,"to":[1,2]}]'
assert b.layout('dsp', 3) == '[{"kind":"grid","container":"dsp","span":3,"grows":"rows","cell_rotation":0.0}]'
assert b.child('dsp', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
assert b.child('dsp', 'b') == '[{"kind":"place","shape":"b","order":1,"to":[0,1]},{"kind":"extents","rows":1,"columns":2}]'
assert b.child('dsp', 'c') == '[{"kind":"place","shape":"c","order":2,"to":[0,2]},{"kind":"extents","rows":1,"columns":3}]'
assert b.child('dsp', 'd') == '[{"kind":"place","shape":"d","order":3,"to":[1,0]},{"kind":"extents","rows":2,"columns":3}]'
assert b.spec('dsp', 'a', 'd', 180.0) == '[{"kind":"rule","from_shape":"a","to_shape":"d","degrees":180.0,"status":"held"}]'
assert b.spec('dsp', 'a', 'b', 270.0) == '[{"kind":"rule","from_shape":"a","to_shape":"b","degrees":270.0,"status":"ignored"}]'
assert b.unspec('dsp', 'a', 'b') == '[{"kind":"rule","from_shape":"a","to_shape":"b","degrees":270.0,"status":"dropped"}]'
assert b.cells("dsp") == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1]]'
assert b.rules("dsp") == '[["a","d",180.0,false]]'
assert text(b.problems("cpu", "gpu", "lsu", "mmu", "io", "dsp")) == '[[],[],[],[],[],[]]'

<a id="CNF-08"></a>
## CNF-08 · The journal, replayed in a fresh store, makes the same adjustments

In [ ]:
assert text(b.replay()) == '[]'